# GCI Global 2026 — Home Credit Default Risk




In [5]:
import os

print(os.listdir('/content'))

['.config', 'competition', 'Competition-20261007T141130Z-1-001.zip', 'sample_data']


In [6]:
import zipfile
import os

zip_path = "/content/Competition-20261007T141130Z-1-001.zip"

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall('/content/competition')

print("ZIP Successfully Extracted!")

ZIP Successfully Extracted!


In [7]:
import glob

print(glob.glob('/content/competition/**/train.csv', recursive=True))

['/content/competition/Competition/input/train.csv']


## 1. Install and import

In [8]:
!pip -q install catboost
import os, glob, zipfile, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

## 2. Locate and load the provided data

In [9]:
# If you extracted the competition ZIP in Colab, this searches for its input folder.
matches = glob.glob("/content/**/input/train.csv", recursive=True)
if not matches:
    raise FileNotFoundError("Could not find input/train.csv. Extract the supplied competition ZIP in Colab first.")
train_path = matches[0]
data_dir = Path(train_path).parent
test_path = data_dir / "test.csv"
sample_path = data_dir / "sample_submission.csv"

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sample_sub = pd.read_csv(sample_path)

print("Train:", train.shape, "Test:", test.shape)
display(train.head())
print("Target distribution:")
display(train["TARGET"].value_counts(normalize=True).rename("share"))

Train: (171202, 34) Test: (61500, 33)


,SK_ID_CURR,NAME_CONTRACT_TYPE,CODE_GENDER,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,...,ORGANIZATION_TYPE,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,DAYS_LAST_PHONE_CHANGE,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR,TARGET
0,0,Cash loans,F,0,112500.0,755190.0,36328.5,675000.0,Working,Higher education,...,School,NaN,0.372591,NaN,-292.0,NaN,NaN,NaN,NaN,0
1,1,Cash loans,F,0,225000.0,585000.0,16893.0,585000.0,Pensioner,Secondary / secondary special,...,XNA,NaN,0.449567,0.553165,-617.0,0.0,0.0,0.0,1.0,0
2,2,Cash loans,F,0,54000.0,334152.0,18256.5,270000.0,State servant,Secondary / secondary special,...,Postal,NaN,0.569503,NaN,-542.0,NaN,NaN,NaN,NaN,0
3,3,Cash loans,F,0,67500.0,152820.0,8901.0,135000.0,Pensioner,Lower secondary,...,XNA,NaN,0.105235,0.767523,0.0,0.0,0.0,0.0,0.0,0
4,4,Cash loans,M,0,157500.0,271066.5,21546.0,234000.0,Commercial associate,Secondary / secondary special,...,Business Entity Type 3,0.342344,0.202490,0.669057,-1243.0,0.0,0.0,0.0,4.0,1


Target distribution:


,share
TARGET,
0,0.919271
1,0.080729


## 3. Basic data checks

## 4. Feature preparation.

In [10]:
assert "TARGET" in train.columns
assert "SK_ID_CURR" in train.columns and "SK_ID_CURR" in test.columns
assert set(test.columns).issubset(set(train.columns))
print("Missing values — top 20:")
display(train.isna().sum().sort_values(ascending=False).head(20))
print("Duplicate training IDs:", train["SK_ID_CURR"].duplicated().sum())

Missing values — top 20:


,0
EXT_SOURCE_1,118928
OWN_CAR_AGE,112992
EXT_SOURCE_3,54586
AMT_REQ_CREDIT_BUREAU_MON,23116
AMT_REQ_CREDIT_BUREAU_QRT,23116
AMT_REQ_CREDIT_BUREAU_HOUR,23116
AMT_REQ_CREDIT_BUREAU_YEAR,23116
EXT_SOURCE_2,369
CNT_FAM_MEMBERS,2
SK_ID_CURR,0


Duplicate training IDs: 0


In [11]:
TARGET = "TARGET"
ID = "SK_ID_CURR"

X = train.drop(columns=[TARGET, ID]).copy()
y = train[TARGET].astype(int).copy()
X_test = test.drop(columns=[ID]).copy()

# Align test columns exactly to training feature columns.
X_test = X_test.reindex(columns=X.columns)

def add_features(df):
    df = df.copy()
    # Convert day counts to interpretable positive years.
    for col in ["DAYS_BIRTH", "DAYS_EMPLOYED", "DAYS_REGISTRATION"]:
        if col in df.columns:
            df[col + "_YEARS"] = df[col].abs() / 365.25
    # Financial ratios; protect against zero denominators.
    if {"AMT_CREDIT", "AMT_INCOME_TOTAL"}.issubset(df.columns):
        df["CREDIT_INCOME_RATIO"] = df["AMT_CREDIT"] / df["AMT_INCOME_TOTAL"].replace(0, np.nan)
    if {"AMT_ANNUITY", "AMT_INCOME_TOTAL"}.issubset(df.columns):
        df["ANNUITY_INCOME_RATIO"] = df["AMT_ANNUITY"] / df["AMT_INCOME_TOTAL"].replace(0, np.nan)
    if {"AMT_CREDIT", "AMT_ANNUITY"}.issubset(df.columns):
        df["CREDIT_ANNUITY_RATIO"] = df["AMT_CREDIT"] / df["AMT_ANNUITY"].replace(0, np.nan)
    if {"AMT_GOODS_PRICE", "AMT_CREDIT"}.issubset(df.columns):
        df["GOODS_CREDIT_RATIO"] = df["AMT_GOODS_PRICE"] / df["AMT_CREDIT"].replace(0, np.nan)
    if {"CNT_CHILDREN", "CNT_FAM_MEMBERS"}.issubset(df.columns):
        df["CHILDREN_PER_FAMILY"] = df["CNT_CHILDREN"] / df["CNT_FAM_MEMBERS"].replace(0, np.nan)
    return df

X = add_features(X)
X_test = add_features(X_test)

# CatBoost categorical columns must be strings; missing categories get an explicit token.
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
for col in cat_cols:
    X[col] = X[col].fillna("__MISSING__").astype(str)
    X_test[col] = X_test[col].fillna("__MISSING__").astype(str)

# Ensure numeric columns are numeric.
for col in X.columns:
    if col not in cat_cols:
        X[col] = pd.to_numeric(X[col], errors="coerce")
        X_test[col] = pd.to_numeric(X_test[col], errors="coerce")

cat_idx = [X.columns.get_loc(c) for c in cat_cols]
print("Features:", X.shape[1], "| Categorical:", len(cat_cols))

Features: 40 | Categorical: 8


## 5. Stratified validation and model training

In [12]:
X_tr, X_va, y_tr, y_va = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)

model = CatBoostClassifier(
    iterations=1400,
    depth=7,
    learning_rate=0.045,
    loss_function="Logloss",
    eval_metric="AUC",
    l2_leaf_reg=5,
    random_seed=SEED,
    verbose=200,
    allow_writing_files=False,
    thread_count=-1
)

model.fit(
    X_tr, y_tr,
    cat_features=cat_idx,
    eval_set=(X_va, y_va),
    early_stopping_rounds=120,
    use_best_model=True
)

valid_pred = model.predict_proba(X_va)[:, 1]
valid_auc = roc_auc_score(y_va, valid_pred)
print(f"Validation ROC-AUC: {valid_auc:.6f}")

0:	test: 0.6754397	best: 0.6754397 (0)	total: 1.03s	remaining: 24m 4s
200:	test: 0.7501093	best: 0.7501472 (199)	total: 1m 30s	remaining: 8m 56s
400:	test: 0.7549388	best: 0.7549484 (399)	total: 2m 41s	remaining: 6m 41s
600:	test: 0.7561694	best: 0.7562453 (548)	total: 4m 14s	remaining: 5m 38s
800:	test: 0.7570775	best: 0.7570775 (800)	total: 5m 26s	remaining: 4m 3s
1000:	test: 0.7574383	best: 0.7574383 (1000)	total: 6m 37s	remaining: 2m 38s
1200:	test: 0.7576777	best: 0.7578937 (1178)	total: 7m 49s	remaining: 1m 17s
Stopped by overfitting detector  (120 iterations wait)

bestTest = 0.7578936533
bestIteration = 1178

Shrink model to first 1179 iterations.
Validation ROC-AUC: 0.757894


## 6. Inspect feature importance

In [13]:
importance = pd.DataFrame({
    "feature": X.columns,
    "importance": model.get_feature_importance()
}).sort_values("importance", ascending=False)
display(importance.head(25))

,feature,importance
25,EXT_SOURCE_2,11.902233
26,EXT_SOURCE_3,9.893100
37,CREDIT_ANNUITY_RATIO,8.701453
38,GOODS_CREDIT_RATIO,4.406215
15,DAYS_ID_PUBLISH,4.244857
27,DAYS_LAST_PHONE_CHANGE,3.603777
24,EXT_SOURCE_1,3.581386
33,DAYS_EMPLOYED_YEARS,3.034150
5,AMT_ANNUITY,2.821754
36,ANNUITY_INCOME_RATIO,2.717909


## 7. Train final model on all training rows and predict test

In [14]:
best_iterations = model.get_best_iteration()
if best_iterations is None or best_iterations <= 0:
    best_iterations = 1400
print("Final iterations:", best_iterations)

final_model = CatBoostClassifier(
    iterations=best_iterations,
    depth=7,
    learning_rate=0.045,
    loss_function="Logloss",
    eval_metric="AUC",
    l2_leaf_reg=5,
    random_seed=SEED,
    verbose=200,
    allow_writing_files=False,
    thread_count=-1
)
final_model.fit(X, y, cat_features=cat_idx)

test_pred = final_model.predict_proba(X_test)[:, 1]
print("Predictions:", len(test_pred), "Min:", test_pred.min(), "Max:", test_pred.max())

Final iterations: 1178
0:	total: 386ms	remaining: 7m 34s
200:	total: 1m 24s	remaining: 6m 49s
400:	total: 2m 43s	remaining: 5m 17s
600:	total: 4m 4s	remaining: 3m 55s
800:	total: 5m 26s	remaining: 2m 33s
1000:	total: 6m 47s	remaining: 1m 12s
1177:	total: 7m 58s	remaining: 0us
Predictions: 61500 Min: 0.0009968264639921133 Max: 0.8339947928958369


## 8. Create and verify submission CSV

In [15]:
submission = sample_sub.copy()
submission["TARGET"] = test_pred
submission = submission[["SK_ID_CURR", "TARGET"]]

assert list(submission.columns) == ["SK_ID_CURR", "TARGET"]
assert len(submission) == len(test)
assert submission["SK_ID_CURR"].equals(test["SK_ID_CURR"])
assert submission["TARGET"].notna().all()
assert submission["TARGET"].between(0, 1).all()

output_dir = data_dir.parent / "output"
output_dir.mkdir(parents=True, exist_ok=True)
submission_path = output_dir / "submission.csv"
submission.to_csv(submission_path, index=False)

print("Saved:", submission_path)
display(submission.head())
print("Submission shape:", submission.shape)

Saved: /content/competition/Competition/output/submission.csv


,SK_ID_CURR,TARGET
0,171202,0.029287
1,171203,0.193741
2,171204,0.170888
3,171205,0.099624
4,171206,0.193597


Submission shape: (61500, 2)


## 9. Download submission from Colab

Run this cell to download `submission.csv` to your computer.

## 10.  code ZIP requirement



In [16]:
from google.colab import files
files.download(str(submission_path))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
# Run after saving this notebook as tutorial.ipynb in the competition folder.
notebook_path = data_dir.parent / "tutorial.ipynb"
zip_path = data_dir.parent / "tutorial.zip"

if notebook_path.exists():
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(notebook_path, arcname="tutorial.ipynb")
    print("Created:", zip_path)
else:
    print("Save/download this notebook as tutorial.ipynb into:", data_dir.parent)
    print("Then rerun this cell.")

Created: /content/competition/Competition/tutorial.zip
